# Motion Correction

1. Creates multi-page tiff and saves it as a single file
2. Applies Median Filter and Gaussian Filter to video
3. Applies Motion Correction using CaImAn


BEFORE YOU BEGIN


This notebook combines the demo_pipeline notebook from CaImAn and some previous code from CSHL2025_basics. To use this notebook, please ensure that you have the caiman conda environment installed. For installation instructions, please reference (https://github.com/flatironinstitute/CaImAn?tab=readme-ov-file#quick-start-route-a)

Some parts of this notebook are meant to be edited and some parts are required to stay as is -- this will be noted in the top line of each cell block.

In [ ]:
# from caiman's demo_pipeline: do not edit
import bokeh.plotting as bpl
import cv2
import datetime
import glob
import holoviews as hv
from IPython import get_ipython
import logging
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import psutil
from pathlib import Path

try:
    cv2.setNumThreads(0)
except():
    pass

try:
    if __IPYTHON__:
        get_ipython().run_line_magic('load_ext', 'autoreload')
        get_ipython().run_line_magic('autoreload', '2')
except NameError:
    pass

import caiman as cm
from caiman.motion_correction import MotionCorrect
from caiman.source_extraction.cnmf import cnmf, params
from caiman.utils.utils import download_demo
from caiman.utils.visualization import plot_contours, nb_view_patches, nb_plot_contour
from caiman.utils.visualization import nb_view_quilt

bpl.output_notebook()
hv.notebook_extension('bokeh')

logfile = None # Replace with a path if you want to log to a file
logger = logging.getLogger('caiman')
# Set to logging.INFO if you want much output, potentially much more output
logger.setLevel(logging.WARNING)
logfmt = logging.Formatter('%(relativeCreated)12d [%(filename)s:%(funcName)20s():%(lineno)s] [%(process)d] %(message)s')
if logfile is not None:
    handler = logging.FileHandler(logfile)
else:
    handler = logging.StreamHandler()
handler.setFormatter(logfmt)
logger.addHandler(handler)

# set env variables 
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"

Custom Functions to make your life easier
- calculating summary images
- saving multi-page tiff files (only for data from ThorLabs system or pre-exiting multi-page)
- filtering videos for denoising



In [ ]:

import numpy as np
import matplotlib.pyplot  as plt
import plotly.express as px
import tifffile as tiff
import os
import imageio as io

from scipy.ndimage import median_filter, gaussian_filter

In [ ]:
def calculate_summary_image(selectSummary, video):
    # selectSummary = "fano"   # Try switching between "mean", "median", "max" and "fano"

    match selectSummary:
        case "mean":
            summaryImage = video.mean(axis=0) 
        case "max":
            summaryImage = video.max(axis=0)
        case "median":
            summaryImage = np.median(video,axis=0)
        case "fano":
            summaryImage = 20*np.var(video,axis=0)/(video.mean(axis=0)*2) #the 20 is to normalize
    return summaryImage


In [ ]:
def save_as_multipage_tif(base_folder, folder, first_file_name, output_folder):

        folder_path = os.path.join(base_folder, folder)
        file_path=  os.path.join(folder_path, first_file_name)
        save_filename = folder.replace(' ', '_') + '_multipage.tif'
        save_path = os.path.join(output_folder,save_filename)

        if not os.path.exists(save_path):
            print(f"Saving multi-page TIFF to: {save_path}")
            apr_mouse1_video = tiff.imread(file_path)
            if apr_mouse1_video.ndim < 3:
                print("The image is not a multi-page tif. Double check that your tif file loads as a multi-page tiff through np.shape(tiff.imread(file_path)).")
            else:
                save_path = os.path.join(folder_path,save_filename)
                tiff.imwrite(save_path, apr_mouse1_video, photometric='minisblack')
        else:
            print(f"File already exists: {save_path}")

In [ ]:
# filtering to make motion correction easier
def filter_video(filter_choice, base_folder, folder, first_file_name, output_folder, kernel_size=3):
    
        folder_path = os.path.join(base_folder, folder)
        file_path=  os.path.join(folder_path, first_file_name)

        apr_mouse1_video = tiff.imread(file_path)
        save_filename = folder.replace(' ', '_') + '_multipage.tif'

        match filter_choice:
            case "median": 
                save_filename = 'medFilt_' + save_filename
                save_path = os.path.join(output_folder,save_filename)
                if not os.path.exists(save_path):
                    filtered_video  = median_filter(apr_mouse1_video, size=kernel_size)
                    print(f"Saving multi-page TIFF to: {save_path}")
                    tiff.imwrite(save_path, filtered_video, photometric='minisblack')
                else:
                    print(f"File already exists: {save_path}")
            case "gaussian":
                save_filename = 'gaussFilt_' + save_filename
                save_path = os.path.join(output_folder,save_filename)
                if not os.path.exists(save_path):
                    filtered_video  = gaussian_filter(apr_mouse1_video,  sigma=kernel_size)
                    print(f"Saving multi-page TIFF to: {save_path}")
                    tiff.imwrite(save_path, filtered_video, photometric='minisblack')
                else:
                    print(f"File already exists: {save_path}")                
             



In [ ]:
# Takes in video, applies filter, saves at specific filepath
def filter_video_directInput(apr_mouse1_video,  filter_choice, output_folder, save_filename, kernel_size=3):
    match filter_choice:
        case "median": 
            save_filename = 'medFilt_' + save_filename
            save_path = os.path.join(output_folder,save_filename)
            if not os.path.exists(save_path):
                filtered_video  = median_filter(apr_mouse1_video, size=kernel_size)
                print(f"Saving multi-page TIFF to: {save_path}")
                tiff.imwrite(save_path, filtered_video, photometric='minisblack')
            else:
                print(f"File already exists: {save_path}")
        case "gaussian":

            save_filename = 'gaussFilt_' + save_filename
            save_path = os.path.join(output_folder,save_filename)
            if not os.path.exists(save_path):
                filtered_video  = gaussian_filter(apr_mouse1_video,  sigma=kernel_size)
                print(f"Saving multi-page TIFF to: {save_path}")
                tiff.imwrite(save_path, filtered_video, photometric='minisblack')
            else:
                print(f"File already exists: {save_path}")  

In [ ]:
def video_normalize(video):
    if video.max() == video.min():
        print("Returning zeros - Video Max/Min is the same value")
        video_normalized = np.zeros_like(video, dtype=np.float32)
    else:
        video_normalized = (video - video.min()) / (video.max() - video.min())
    return video_normalized

## Save multipage tiffs and also filtered videos

In [ ]:
# save tiffs of interest as a multi-page tif
# Note that this is mainly if you're saving as single tifs and you got data from a thorlabs system
# If you already have a multi-page tiff, utilize the commented-out code at the bottom of the cell block
 
# here I was doing for multiple folders, you can do individual ones as well
base_folder = '/home/estherwhang/cshl2025/sergey_data/8.2.2025_28X_test_in-vivo/'
folder_name = 'APR mouse 1 25X'
output_folder='/home/estherwhang/cshl2025/sergey_data/results' # where to find your outputs
    
first_file_name = 'ChanA_001_001_001_001.tif'
save_as_multipage_tif(base_folder, folder_name, first_file_name, output_folder)
filter_video('median', base_folder, folder_name,  first_file_name, output_folder)
filter_video('gaussian', base_folder, folder_name,  first_file_name, output_folder)

# if you already have a multi-page tif you want to use directly, use the following code
# I do suggest renaming the mutl-page tif to whatever you set the save_filename variable to,
# as it will make the rest of the notebook much easier

# file_name = folder.replace(' ', '_')
# apr_mouse1_video = tiff.imread(file_path_to_video)
# filter_video_directInput(apr_mouse1_video,  filter_choice, output_folder, save_filename)

Let's check the outputs of the original video and the filtered data

In [ ]:
# # first, get some correlation and max projection images:
# import plotly.express as px
# gauss_check = tiff.imread('/home/estherwhang/cshl2025/sergey_data/summary_images/max_proj_APR_mouse_1_25X.tif')

# px.imshow(gauss_check, title="Original, Median, and Gaussian Filtered")

# Let's check that you saved everything correctly! 

First, we will look at the videos for the original and different filtered videos

We will check the maximum projection and the correlation image to make sure we have the right videos

Lastly, we'll look at the distribution of values (for fun) and a few extra summary images (for fun)

In [ ]:
# check what we're loading in
import plotly.express as px

folder = 'APR mouse 1 25X'
filename = folder.replace(' ', '_')

save_filename = filename + '_multipage.tif'
multipage_path_orig = os.path.join(output_folder, save_filename)
apr_mouse1_video = tiff.imread(multipage_path_orig)

save_filename = 'medFilt_'+ filename + '_multipage.tif'
multipage_path_medFilt = os.path.join(output_folder, save_filename)
medFilt_apr_mouse1_video = tiff.imread(multipage_path_medFilt)

save_filename = 'gaussFilt_'+filename + '_multipage.tif'
multipage_path_gaussFilt = os.path.join(output_folder, save_filename)
gaussFilt_apr_mouse1_video = tiff.imread(multipage_path_gaussFilt)

frame_subset = 100

compare_videos =  np.concatenate([video_normalize(apr_mouse1_video[0:frame_subset]), 
                                  video_normalize(medFilt_apr_mouse1_video[0:frame_subset]),
                                  video_normalize(gaussFilt_apr_mouse1_video[0:frame_subset])], axis=2)


import plotly.express as px
fig = px.imshow(compare_videos,  animation_frame=0, binary_string=True, title="Original, Median, and Gaussian Filtered", labels=dict(animation_frame="slice"))
fig.show()

In [ ]:
# let's just look at the original movie

# let's load it using the caiman package, which requires the file path
movie = cm.load(multipage_path_orig) 

# calculate the maximum projection and local correlation images
max_projection = np.max(movie, axis=0)
correlation_image = cm.local_correlations(movie, swap_dim=False)
correlation_image[np.isnan(correlation_image)] = 0 # get rid of NaNs, if they exist

In [ ]:
# Let's plot them
f, (ax_max, ax_corr) = plt.subplots(1,2,figsize=(6,3))
ax_max.imshow(max_projection, 
              cmap='viridis',
              vmin=np.percentile(np.ravel(max_projection),50), 
              vmax=np.percentile(np.ravel(max_projection),99.5));
ax_max.set_title("Max Projection Orig", fontsize=12);

ax_corr.imshow(correlation_image, 
               cmap='viridis', 
               vmin=np.percentile(np.ravel(correlation_image),50), 
               vmax=np.percentile(np.ravel(correlation_image),99.5));
ax_corr.set_title('Correlation Image Orig', fontsize=12);


In [ ]:
# save the output images
save_maxproj_filename = 'max_proj_' + filename + '.tif'
io.imwrite(os.path.join(output_folder,save_maxproj_filename), max_projection)

save_corr_filename = 'corr_' + filename + '.tif'
io.imwrite(os.path.join(output_folder,save_corr_filename), correlation_image)

In [ ]:

number_of_bins = 100 # determines the number of bins in the histogram
plt.hist(apr_mouse1_video.ravel(),bins=number_of_bins, label="Original")
plt.hist(medFilt_apr_mouse1_video.ravel(),bins=number_of_bins, label="Median Filter")
plt.hist(gaussFilt_apr_mouse1_video.ravel(),bins=number_of_bins, label="Gaussian Filter")
# labels
plt.title("Histogram of Intensity Values for Three Videos")
plt.ylabel("Count")
plt.xlabel("Intensity Value")
plt.legend()


In [ ]:
# Also a good idea to look at some of the summary images

summaryImage_mean = calculate_summary_image('mean', apr_mouse1_video)
summaryImage_fano = calculate_summary_image('fano', apr_mouse1_video)

plt.subplot(1,2,1), plt.imshow(summaryImage_mean), plt.title("Mean of Original")
color_bar = plt.colorbar()
color_bar.set_label('Intensity')
plt.subplot(1,2,2), plt.imshow(summaryImage_fano), plt.title("Fano of Original")
color_bar = plt.colorbar()
color_bar.set_label('Intensity')


# Using CaImAn with Filtered Data

CaImAn has a very specific way it wants you to load in data which isn't the most intuitive - play close attention to file paths in the following section.

In [ ]:
# output_folder = whereever you put the multi-page tif you want to reconstruct

folder = 'APR mouse 1 25X' 
filename = folder.replace(' ', '_') + '_multipage.tif'

# uncomment the following for whatever verion you want

## "original":
# save_filename = file_name 
# fnames = os.path.join(output_folder, save_filename)
## "median":
# save_filename = 'medFilt_'+ file_name
# fnames = os.path.join(output_folder, save_filename)
## "gaussian":
save_filename = 'gaussFilt_'+ filename 
fnames = os.path.join(output_folder, save_filename)
print(save_filename)

In [ ]:
# check the file path
print(fnames)

# Motion correction stuff from CaImAn

In [ ]:
# Parameters from CaImAn
max_shifts = (6, 6)  # maximum allowed rigid shift in pixels (view the movie to get a sense of motion)
strides =  (48, 48)  # create a new patch every x pixels for pw-rigid correction
overlaps = (24, 24)  # overlap between patches (size of patch strides+overlaps)
max_deviation_rigid = 3   # maximum deviation allowed for patch with respect to rigid shifts
pw_rigid = False  # flag for performing rigid or piecewise rigid motion correction
shifts_opencv = True  # flag for correcting motion using bicubic interpolation (otherwise FFT interpolation is used)
border_nan = 'copy'  # replicate values along the boundary (if True, fill in with NaN)

In [ ]:
#%% start the cluster (if a cluster already exists terminate it)
if 'dview' in locals():
    cm.stop_server(dview=dview)
c, dview, n_processes = cm.cluster.setup_cluster(
    backend='multiprocessing', n_processes=None, single_thread=False)

In [ ]:
# create a motion correction object
mc = MotionCorrect(fnames, dview=dview, max_shifts=max_shifts,
                  strides=strides, overlaps=overlaps,
                  max_deviation_rigid=max_deviation_rigid, 
                  shifts_opencv=shifts_opencv, nonneg_movie=True,
                  border_nan=border_nan)

In [ ]:
%%capture
# correct for rigid motion correction and save the file (in memory mapped form)
mc.motion_correct(save_movie=True)

In [ ]:
# load motion corrected movie
m_rig = cm.load(mc.mmap_file)
bord_px_rig = np.ceil(np.max(mc.shifts_rig)).astype(int)
#%% visualize templates
plt.figure(figsize = (20,10))
plt.imshow(mc.total_template_rig, cmap = 'gray');

In [ ]:
plt.close()
plt.figure(figsize = (20,10))
plt.plot(mc.shifts_rig)
plt.legend(['x shifts','y shifts'])
plt.xlabel('frames')
plt.ylabel('pixels');

In [ ]:
# saving data
NoRMCorre_save_filename = 'NoRMCorre_'+ save_filename 
# save_filename = 'gaussFilt_' + save_filename
NoRMCorre_fnames = os.path.join(output_folder,NoRMCorre_save_filename)
m_rig.save(NoRMCorre_fnames)
print(NoRMCorre_fnames)

## checking data

In [ ]:
# loading 
folder = 'APR mouse 1 25X'

NoRMCorre_gaussFilt_save_filename = 'NoRMCorre_'+ folder.replace(' ', '_') + '_multipage.tif'
# save_filename = 'gaussFilt_' + save_filename
NoRMCorre_gaussFilt_fnames = os.path.join(output_folder,NoRMCorre_gaussFilt_save_filename)
check1 = tiff.imread(NoRMCorre_gaussFilt_fnames)


NoRMCorre_gaussFilt_save_filename = 'NoRMCorre_gaussFilt_'+ folder.replace(' ', '_') + '_multipage.tif'
# save_filename = 'gaussFilt_' + save_filename
NoRMCorre_gaussFilt_fnames = os.path.join(output_folder,NoRMCorre_gaussFilt_save_filename)
check2 = tiff.imread(NoRMCorre_gaussFilt_fnames)


NoRMCorre_gaussFilt_save_filename = 'NoRMCorre_medFilt_'+ folder.replace(' ', '_') + '_multipage.tif'
# save_filename = 'gaussFilt_' + save_filename
NoRMCorre_gaussFilt_fnames = os.path.join(output_folder,NoRMCorre_gaussFilt_save_filename)
check3 = tiff.imread(NoRMCorre_gaussFilt_fnames)


summaryImage_mean_1 = calculate_summary_image('mean', check1)
summaryImage_mean_2 = calculate_summary_image('mean', check2)
summaryImage_mean_3 = calculate_summary_image('mean', check3)


plt.subplot(1,3,1), plt.imshow(summaryImage_mean_1), plt.title("Original")
color_bar = plt.colorbar()
color_bar.set_label('Intensity')
plt.subplot(1,3,2), plt.imshow(summaryImage_mean_2), plt.title("Median Filtered")
color_bar = plt.colorbar()
color_bar.set_label('Intensity')
plt.subplot(1,3,3), plt.imshow(summaryImage_mean_3), plt.title("Gaussian Filtered")
color_bar = plt.colorbar()
color_bar.set_label('Intensity')
